# Build Model

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from datetime import datetime as dt
import bentoml
from joblib import load
import os


pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)

In [2]:
from dota_oracle_common.postgresql import DatabaseEngineFactory
from sqlalchemy.ext.asyncio import AsyncSession

engine = DatabaseEngineFactory.get_engine()
engine

In [3]:
from dota_oracle_common.repositories.match_repository import MatchRepository


async with AsyncSession(engine) as session:
    match_repo = MatchRepository(session)
    
    matches = await match_repo.get_match_details(
        relationship_fields=[
            "outcome", "team_features", "player_hero_features", "hero_features"
        ]
    )
    
    print(f"number of matches: {len(matches)}")
    



dota_oracle_common.repositories.base_repository - Retrieved 96821 records for MatchTable
dota_oracle_common.repositories.match_repository - Found 96821 MatchTable details.


number of matches: 96821


In [4]:
match_outcome_list = []
hero_features_list = []
player_hero_team_features_list = []

for match in matches:
    outcome = match.outcome
    match_outcome_list.append(outcome.model_dump())
    
    team_features = match.team_features
    hero_features = match.hero_features
    player_hero_features = match.player_hero_features
    
    player_hero_team_features_dict = {**team_features.model_dump(), **player_hero_features.model_dump()}
    
    hero_features_list.append(hero_features.model_dump())
    player_hero_team_features_list.append(player_hero_team_features_dict)
    

print(f"count match_outcome_list: {len(match_outcome_list)}")
print(f"count features_list: {len(player_hero_team_features_list)}")

count match_outcome_list: 96821
count features_list: 96821


In [5]:
outcome_df = pd.DataFrame(match_outcome_list)
player_hero_team_df = pd.DataFrame(player_hero_team_features_list)
hero_df = pd.DataFrame(hero_features_list)

In [6]:
outcome_df

,match_id,radiant_win
0,8230722475,True
1,8230701740,False
2,8230693148,True
3,8230677659,True
4,8230656847,True
...,...,...
96816,5999283181,False
96817,5999249937,False
96818,5999214195,False
96819,5999201501,True


In [7]:
player_hero_team_df

,match_id,radiant_dire_matchup,radiant_win_rate,dire_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_4_win_rate,player_hero_129_win_rate,player_hero_131_win_rate,player_hero_3_win_rate,player_hero_0_win_rate,player_hero_128_win_rate,player_hero_130_win_rate,player_hero_132_win_rate
0,8230722475,0.333333,0.5,0.7,0.550000,0.600000,0.4,0.769231,0.500000,0.75,0.600000,0.800000,0.714286,0.80
1,8230701740,0.600000,0.7,0.5,0.500000,0.384615,0.4,0.666667,0.550000,0.70,0.642857,0.300000,0.300000,0.50
2,8230693148,0.625000,0.6,0.6,0.714286,0.692308,0.5,0.500000,0.833333,1.00,0.777778,0.588235,0.600000,0.50
3,8230677659,0.350000,0.4,0.8,0.350000,0.625000,0.6,0.600000,1.000000,0.45,0.250000,0.625000,1.000000,0.00
4,8230656847,0.600000,0.7,0.4,0.538462,0.400000,0.5,0.350000,0.600000,0.40,0.450000,0.384615,0.500000,0.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,5999283181,0.500000,0.5,0.5,0.500000,0.500000,0.5,0.500000,0.500000,0.50,0.500000,0.500000,0.500000,0.50
96817,5999249937,1.000000,1.0,0.0,0.500000,0.500000,0.5,0.000000,0.500000,1.00,0.500000,0.500000,0.500000,0.50
96818,5999214195,0.500000,0.5,0.5,0.500000,0.500000,0.5,0.500000,0.500000,0.50,0.500000,0.500000,0.500000,0.50
96819,5999201501,0.500000,0.5,0.5,0.500000,0.500000,0.5,0.500000,0.500000,0.50,0.500000,0.500000,0.500000,0.50


In [8]:
hero_df

,hero_picks,match_id
0,"[Sven, Tiny, Clockwerk, Shadow Demon, Dark See...",8230722475
1,"[Ancient Apparition, Magnus, Pudge, Puck, Sven...",8230701740
2,"[Tinker, Pangolier, Tiny, Leshrac, Lifestealer...",8230693148
3,"[Zeus, Pudge, Wraith King, Sniper, Death Proph...",8230677659
4,"[Jakiro, Tidehunter, Lina, Rubick, Tiny, Anti-...",8230656847
...,...,...
96816,"[Enchantress, Timbersaw, Tiny, Wraith King, Or...",5999283181
96817,"[Void Spirit, Brewmaster, Terrorblade, Snapfir...",5999249937
96818,"[Centaur Warrunner, Hoodwink, Razor, Grimstrok...",5999214195
96819,"[Ancient Apparition, Enchantress, Magnus, Embe...",5999201501


In [9]:
from dota_oracle_pipeline.feature_transformation.feature_encoder import FeatureEncoder
from dota_oracle_common.repositories.heroes_repository import HeroesRepository

async with AsyncSession(engine) as session:
    heros_repo = HeroesRepository(session)
    hero_map = await heros_repo.get_hero_id_map()

encoded_hero_features = FeatureEncoder.encode_hero_features(hero_features=hero_df, hero_map=hero_map)

In [10]:
encoded_hero_features

,Pugna,Anti-Mage,Axe,Bane,Bloodseeker,Crystal Maiden,Drow Ranger,Earthshaker,Juggernaut,Mirana,Morphling,Shadow Fiend,Phantom Lancer,Puck,Pudge,Razor,Sand King,Storm Spirit,Sven,Tiny,Vengeful Spirit,Windranger,Zeus,Kunkka,Lina,Lion,Shadow Shaman,Slardar,Tidehunter,Witch Doctor,Lich,Riki,Enigma,Tinker,Sniper,Necrophos,Warlock,Beastmaster,Queen of Pain,Venomancer,Faceless Void,Wraith King,Death Prophet,Phantom Assassin,Templar Assassin,Viper,Luna,Dragon Knight,Dazzle,Clockwerk,...,Shadow Demon,Lone Druid,Chaos Knight,Meepo,Treant Protector,Ogre Magi,Undying,Rubick,Disruptor,Nyx Assassin,Naga Siren,Keeper of the Light,Io,Visage,Slark,Medusa,Troll Warlord,Centaur Warrunner,Magnus,Timbersaw,Bristleback,Tusk,Skywrath Mage,Abaddon,Elder Titan,Legion Commander,Techies,Ember Spirit,Earth Spirit,Underlord,Terrorblade,Phoenix,Oracle,Winter Wyvern,Arc Warden,Monkey King,Dark Willow,Pangolier,Grimstroke,Hoodwink,Void Spirit,Snapfire,Mars,Ring Master,Dawnbreaker,Marci,Primal Beast,Muerta,Kez,match_id
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230722475
1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230701740
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,8230693148
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230677659
4,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230656847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,5999283181
96817,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,5999249937
96818,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,5999214195
96819,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5999201501


In [11]:
merged_df = pd.merge(outcome_df, (pd.merge(player_hero_team_df, encoded_hero_features, how='inner')), how='inner')
merged_df

,match_id,radiant_win,radiant_dire_matchup,radiant_win_rate,dire_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_4_win_rate,player_hero_129_win_rate,player_hero_131_win_rate,player_hero_3_win_rate,player_hero_0_win_rate,player_hero_128_win_rate,player_hero_130_win_rate,player_hero_132_win_rate,Pugna,Anti-Mage,Axe,Bane,Bloodseeker,Crystal Maiden,Drow Ranger,Earthshaker,Juggernaut,Mirana,Morphling,Shadow Fiend,Phantom Lancer,Puck,Pudge,Razor,Sand King,Storm Spirit,Sven,Tiny,Vengeful Spirit,Windranger,Zeus,Kunkka,Lina,Lion,Shadow Shaman,Slardar,Tidehunter,Witch Doctor,Lich,Riki,Enigma,Tinker,Sniper,...,Brewmaster,Shadow Demon,Lone Druid,Chaos Knight,Meepo,Treant Protector,Ogre Magi,Undying,Rubick,Disruptor,Nyx Assassin,Naga Siren,Keeper of the Light,Io,Visage,Slark,Medusa,Troll Warlord,Centaur Warrunner,Magnus,Timbersaw,Bristleback,Tusk,Skywrath Mage,Abaddon,Elder Titan,Legion Commander,Techies,Ember Spirit,Earth Spirit,Underlord,Terrorblade,Phoenix,Oracle,Winter Wyvern,Arc Warden,Monkey King,Dark Willow,Pangolier,Grimstroke,Hoodwink,Void Spirit,Snapfire,Mars,Ring Master,Dawnbreaker,Marci,Primal Beast,Muerta,Kez
0,8230722475,True,0.333333,0.5,0.7,0.550000,0.600000,0.4,0.769231,0.500000,0.75,0.600000,0.800000,0.714286,0.80,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,8230701740,False,0.600000,0.7,0.5,0.500000,0.384615,0.4,0.666667,0.550000,0.70,0.642857,0.300000,0.300000,0.50,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,8230693148,True,0.625000,0.6,0.6,0.714286,0.692308,0.5,0.500000,0.833333,1.00,0.777778,0.588235,0.600000,0.50,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
3,8230677659,True,0.350000,0.4,0.8,0.350000,0.625000,0.6,0.600000,1.000000,0.45,0.250000,0.625000,1.000000,0.00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,8230656847,True,0.600000,0.7,0.4,0.538462,0.400000,0.5,0.350000,0.600000,0.40,0.450000,0.384615,0.500000,0.65,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,5999283181,False,0.500000,0.5,0.5,0.500000,0.500000,0.5,0.500000,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
96817,5999249937,False,1.000000,1.0,0.0,0.500000,0.500000,0.5,0.000000,0.500000,1.00,0.500000,0.500000,0.500000,0.50,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0
96818,5999214195,False,0.500000,0.5,0.5,0.500000,0.500000,0.5,0.500000,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0
96819,5999201501,True,0.500000,0.5,0.5,0.500000,0.500000,0.5,0.500000,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0,0,0,0,0,0,1,0,0,0,0,0,0,

In [12]:
df_final = merged_df.drop('match_id', axis=1)

In [13]:
# Train Test Split
X = df_final.drop('radiant_win', axis=1)
y = df_final['radiant_win']

n_total = len(df_final)
n_test = int(n_total * 0.3)

# Top 30% as test, remaining 70% as train
X_test = X.iloc[:n_test]     # First 30%
X_train = X.iloc[n_test:]    # Remaining 70%
y_test = y.iloc[:n_test]     # First 30%
y_train = y.iloc[n_test:]    # Remaining 70

In [14]:
# Import candidate classifiers

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# import metrics

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [15]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000  # Increase if convergence issues
    ),
    
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1  # Use all cores
    ),
    
    'XGBoost': xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',  # Suppress warning
        n_estimators=100
    ),
    
    'LightGBM': lgb.LGBMClassifier(
        random_state=42,
        verbosity=-1,  # Suppress output
        n_estimators=100
    )
}

In [16]:
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]  # For ROC-AUC
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"{name} Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")


Training Logistic Regression...


/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:919: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:919: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:919: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:919: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(


Logistic Regression Accuracy: 0.553 (55.3%)

Training Random Forest...


/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:919: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:919: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(


Random Forest Accuracy: 0.549 (54.9%)

Training XGBoost...


/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/xgboost/data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/xgboost/data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/xgboost/data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


XGBoost Accuracy: 0.544 (54.4%)

Training LightGBM...
LightGBM Accuracy: 0.556 (55.6%)


In [17]:
clf = models['Random Forest']
clf

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
# Prediction with bentoml model service
import requests
import json
import bentoml

In [19]:
from dota_oracle_common.models.inference.schema import VersionMetaData, PerformanceMetrics, ModelMetaData 

def save_sklearn_model(
    model,
    model_name: str,
    metadata: ModelMetaData
):
    print(f"attempting to save model {model_name}")
    saved_model = bentoml.sklearn.save_model(
        name=model_name,
        model=model, 
        signatures={'predict':{'batchable':True}},
        metadata=metadata.model_dump()
    )
    
    print(f"Model saved: {saved_model}")

In [21]:
model_name = 'dota_oracle_random_forest'

feature_columns = clf.feature_names_in_.tolist()

performance = PerformanceMetrics(
    accuracy=0.551
)

version_data = VersionMetaData(
    performance_metrics=performance,
    feature_columns=feature_columns
)

metadata = metadata = ModelMetaData(
        name=model_name,
        version='0.0.1',
        trained_date=dt.now(),
        version_metadata=version_data
)

try:
    save_sklearn_model(model=clf, model_name=model_name, metadata=metadata)
    print("process complete")
except Exception as e:
    print(f"error saving model to bentoml, {e}")

attempting to save model dota_oracle_random_forest
Model saved: Model(tag="dota_oracle_random_forest:uxznpxsku2u2lryn")
process complete
